In [ ]:
#
# ⚡ UNIVERSAL FIRST CELL - Run this FIRST in every notebook!
# Compatible with: Google Colab, GitHub Codespaces, Local
#

import os
import subprocess
import sys

# Detect environment
IS_COLAB = "google.colab" in sys.modules
IS_CODESPACES = os.path.exists("/.devcontainer") or os.path.exists("/workspaces")
IS_LOCAL = not (IS_COLAB or IS_CODESPACES)

print(f"\U0001f680 Environment: {'Colab' if IS_COLAB else 'Codespaces' if IS_CODESPACES else 'Local'}")

# Ensure we are in the root directory for relative paths
while not os.path.exists('data') and os.path.dirname(os.getcwd()) != os.getcwd():
    os.chdir('..')
print(f"\U0001f4c1 Working directory set to: {os.getcwd()}")

# Install article extraction libraries (only needed on Colab/Codespaces)
if not IS_LOCAL:
    !pip install trafilatura newspaper3k lxml_html_clean -q

print("\u2705 Environment ready!")

# 13 - Article Extraction for Buildout Candidates

**Goal**: Fetch article text for each candidate URL from GDELT GKG and extract structured data center buildout event data.

**Pipeline**:
1. Load candidate URLs from `data/raw/buildout_candidates_gkg.csv`
2. Fetch article text via `trafilatura` (primary) / `newspaper3k` (fallback)
3. Extract: company, location (city+state), MW capacity, dates
4. Classify: is_buildout (bool), confidence (low/medium/high)
5. Save structured events to `data/raw/buildout_events_raw.csv`

**Output columns**: url, source_domain, date, company, location_city, location_state, mw_capacity, target_completion_date, is_buildout, confidence, extracted_text_snippet, v2_organizations, v2_locations, v2_tone

In [ ]:
import pandas as pd
import numpy as np
import re
import json
import time
import warnings
from urllib.parse import urlparse
from datetime import datetime
from pathlib import Path
warnings.filterwarnings('ignore')

# Article text extraction
try:
    import trafilatura
    HAS_TRAFILATURA = True
    print("\u2705 trafilatura available (F1=0.910)")
except ImportError:
    HAS_TRAFILATURA = False
    print("\u26a0\ufe0f trafilatura not installed (pip install trafilatura)")

try:
    from newspaper import Article
    HAS_NEWSPAPER = True
    print("\u2705 newspaper3k available (F1=0.762)")
except ImportError:
    HAS_NEWSPAPER = False
    print("\u26a0\ufe0f newspaper3k not installed (pip install newspaper3k)")

if not (HAS_TRAFILATURA or HAS_NEWSPAPER):
    print("\u274c No article extraction library available! Install trafilatura or newspaper3k.")
    raise SystemExit(0)

print("\u2705 Libraries imported")

## Step 1: Load Candidate URLs

Load the GDELT GKG candidate data produced by `12-gdelt-domain-filter.ipynb`.

Expected columns: DATE, SourceCommonName, DocumentIdentifier, V2Organizations, V2Locations, V2Tone, matched_companies

In [ ]:
# Load candidate data
INPUT_PATH = 'data/raw/buildout_candidates_gkg.csv'

if not os.path.exists(INPUT_PATH):
    print(f"\u274c Input file not found: {INPUT_PATH}")
    print("   Run 12-gdelt-domain-filter.ipynb first to generate candidate data.")
    raise SystemExit(0)

df = pd.read_csv(INPUT_PATH)
print(f"\U0001f4c2 Loaded {len(df)} candidate rows from {INPUT_PATH}")
print(f"   Columns: {list(df.columns)}")

# Verify DocumentIdentifier (URL) column exists
url_col = None
for col in ['DocumentIdentifier', 'url']:
    if col in df.columns:
        url_col = col
        break

if url_col is None:
    print("\u274c No URL column found (expected 'DocumentIdentifier' or 'url')")
    raise SystemExit(0)

print(f"\U0001f310 URL column: '{url_col}'")

# Drop duplicates by URL
n_before = len(df)
df = df.drop_duplicates(subset=[url_col])
n_dupes = n_before - len(df)
print(f"   Removed {n_dupes} duplicate URLs")

# Show domain breakdown
domain_col = 'SourceCommonName' if 'SourceCommonName' in df.columns else 'source_domain'
if domain_col in df.columns:
    print(f"\n\U0001f4cb Domain breakdown:")
    print(df[domain_col].value_counts().to_string())

print(f"\n\U0001f4c8 Unique URLs to process: {len(df)}")

## Step 2: Define Extraction Functions

Implement multi-stage extraction:
1. Fetch article HTML \u2192 extract text with trafilatura (primary) / newspaper3k (fallback)
2. Parse MW capacity with regex patterns
3. Parse location from V2Locations + text regex
4. Identify company from V2Organizations
5. Build confidence score

In [ ]:
# MW capacity regex patterns
# Covers: "200MW", "200 MW", "200-megawatt", "200 megawatts", "200 mw"
MW_PATTERNS = [
    r'(\d{1,5}(?:[.,]\d{1,2})?)\s*-?\s*MW',          # 200MW, 200 MW, 200-MW
    r'(\d{1,5}(?:[.,]\d{1,2})?)\s*megawatts?',         # 200 megawatts
    r'(\d{1,5}(?:[.,]\d{1,2})?)\s*-?\s*megawatt',     # 200-megawatt
    r'(\d{1,5}(?:[.,]\d{1,2})?)\s*mw',                # 200 mw (lowercase)
    r'(\d{1,2})\s*(?:GW|gigawatts?)',                  # GW conversion (e.g. 1.2GW)
]

# US city/state pattern
LOCATION_PATTERN = r'([A-Z][a-z]+(?:\s[A-Z][a-z]+)*)\s*,\s*([A-Z]{2})'

# Buildout-related keywords (positive indicators)
BUILDOUT_KEYWORDS = [
    r'\bdata center\b', r'\bdatacenter\b',
    r'\bbuild\b', r'\bconstruction\b', r'\bbreak\s*ground\b',
    r'\bcampus\b', r'\bfacility\b', r'\bexpansion\b',
    r'\binvestment\b', r'\bcapex\b', r'\bcapital\s*expenditure\b',
    r'\bmegawatt\b', r'\bcapacity\b',
    r'\bannounce\b', r'\bplan\b', r'\bunveil\b',
    r'\bnew\s*(?:data|cloud|AI|server)\s*(?:center|campus|facility|region)\b',
]

# Exclusion keywords (negative indicators - noise, not a buildout)
EXCLUSION_KEYWORDS = [
    r'\blayoff\b', r'\bquit\b', r'\bresign\b',
    r'\blawsuit\b', r'\blitigation\b',
    r'\bclass\s*action\b', r'\bSEC\b',
    r'\bfire\b', r'\bwildfire\b', r'\bflood\b',
    r'\bearthquake\b', r'\bdisaster\b',
]

# Target companies (from 12-gdelt-domain-filter.ipynb)
COMPANY_KEYWORDS = [
    # Hyperscalers
    'Microsoft', 'Microsoft Corp',
    'Google', 'Alphabet',
    'Amazon', 'AWS', 'Amazon Web Services',
    'Meta', 'Facebook',
    'NVIDIA', 'Nvidia',
    'Apple',
    # Enterprise cloud
    'Oracle',
    # GPU cloud
    'Crusoe',
    # Colo / digital infra REITs
    'Equinix',
    'Digital Realty',
    'American Tower',
    'Prologis',
    'Simon Property',
    'Public Storage',
    'Outfront',
    'Sabra',
    'Hudson Pacific',
    'Rexford',
    'First Industrial',
    'SITC',
]

print(f"\u2705 {len(MW_PATTERNS)} MW patterns, {len(BUILDOUT_KEYWORDS)} buildout keywords, {len(EXCLUSION_KEYWORDS)} exclusion keywords")
print(f"\U0001f3af Tracking {len(COMPANY_KEYWORDS)} company patterns")

In [ ]:
def extract_article_text(url, timeout=15):
    """
    Fetch article HTML and extract text.
    Primary: trafilatura (F1=0.910)
    Fallback: newspaper3k (F1=0.762)
    Returns (text, method) or (None, None) on failure.
    """
    method = None
    text = None

    # Primary: trafilatura
    if HAS_TRAFILATURA:
        try:
            downloaded = trafilatura.fetch_url(url, timeout=timeout)
            if downloaded:
                text = trafilatura.extract(downloaded, output_format='text',
                                           include_links=False, include_images=False,
                                           include_tables=False)
                if text and len(text.strip()) > 100:
                    method = 'trafilatura'
                    return text.strip(), method
        except Exception as e:
            pass  # Fall through to newspaper3k

    # Fallback: newspaper3k
    if HAS_NEWSPAPER:
        try:
            article = Article(url, timeout=timeout)
            article.download()
            article.parse()
            if article.text and len(article.text.strip()) > 100:
                text = article.text.strip()
                method = 'newspaper3k'
                return text, method
        except Exception as e:
            pass

    return None, None


def extract_mw(text):
    """Extract MW capacity from article text. Returns float or None."""
    if not text:
        return None

    text_lower = text.lower()
    found_values = []

    # Check for GW pattern first (convert to MW)
    gw_match = re.search(r'(\d{1,2}(?:\.\d{1,2})?)\s*(?:GW|gigawatts?)', text_lower)
    if gw_match:
        val = float(gw_match.group(1).replace(',', '')) * 1000
        found_values.append(val)

    # Standard MW patterns
    for pattern in MW_PATTERNS:
        matches = re.findall(pattern, text_lower, re.IGNORECASE)
        for m in matches:
            # Skip if it looks like a year or date
            try:
                val = float(m.replace(',', ''))
                if 1 <= val <= 50000:  # Sanity check: 1MW to 50GW
                    found_values.append(val)
            except ValueError:
                continue

    if not found_values:
        return None

    # Return the max value (most likely the total buildout capacity)
    return max(found_values)


def extract_location(text, v2_locations=None):
    """Extract US city and state from text and/or V2Locations field.
    Returns (city, state) or (None, None).
    """
    city, state = None, None

    # Strategy 1: Parse V2Locations field
    # Format: "lat,long|name|country|admin1|city" (semicolon-separated)
    if v2_locations and isinstance(v2_locations, str) and v2_locations.strip():
        try:
            for loc_entry in v2_locations.split(';'):
                parts = loc_entry.strip().split('|')
                if len(parts) >= 5:
                    city = parts[4] if parts[4] and parts[4] != 'None' else None
                    state = parts[3] if parts[3] and parts[3] != 'None' else None
                    country = parts[2] if len(parts) > 2 else ''
                    # Only accept US locations
                    if country.strip().upper() in ('US', 'UNITED STATES', 'USA', ''):
                        if city and state:
                            # Clean state code (keep only the state abbreviation part)
                            state_clean = state.split(',')[0].strip()
                            if len(state_clean) == 2:
                                return city, state_clean
        except Exception:
            pass

    # Strategy 2: Regex from text
    if text:
        matches = re.findall(LOCATION_PATTERN, text)
        if matches:
            # Take the last match (often most specific: city, ST)
            city, state = matches[-1]
            return city.strip(), state.strip()

    return None, None


def extract_company(v2_organizations=None, text=None):
    """Identify which target company is mentioned.
    Uses V2Organizations field first, then text fallback.
    """
    found = []

    # Strategy 1: V2Organizations (most reliable)
    if v2_organizations and isinstance(v2_organizations, str):
        orgs_lower = v2_organizations.lower()
        for kw in COMPANY_KEYWORDS:
            if kw.lower() in orgs_lower:
                found.append(kw)

    # Strategy 2: Text fallback if V2 didn't find anything
    if not found and text:
        text_lower = text.lower()
        for kw in COMPANY_KEYWORDS:
            if kw.lower() in text_lower:
                found.append(kw)

    if not found:
        return None

    # Prioritize: longer match wins (more specific)
    found.sort(key=len, reverse=True)
    return found[0]


def extract_target_completion_date(text):
    """Extract target completion/operational date from text.
    Returns date string or None.
    """
    if not text:
        return None

    # Patterns for target completion dates
    date_patterns = [
        r'(?:target|expected|planned|scheduled|to be (?:operational|complete|ready)|by)\s*(?:for\s*)?(?:completion|operational|opening|launch)?\s*:?\s*(\d{4})',
        r'(\d{4})\s*(?:target|expected|planned|scheduled)',
        r'(?:open|launch|complete|operational|ready)\s*(?:in|by)\s*(\d{4})',
        r'(?:Q[1-4]\s*\d{4})',  # e.g., Q1 2026, Q3 2025
        r'(?:H[12]\s*\d{4})',   # e.g., H2 2025
    ]

    text_lower = text.lower()
    for pattern in date_patterns:
        match = re.search(pattern, text_lower, re.IGNORECASE)
        if match:
            return match.group(0).strip()

    return None


def classify_buildout(text, company, mw_capacity):
    """
    Classify whether this article is a buildout announcement.
    Returns (is_buildout: bool, confidence: str).
    """
    if not text:
        return False, 'fetch_failed'

    text_lower = text.lower()

    # Check exclusion keywords first
    for pattern in EXCLUSION_KEYWORDS:
        if re.search(pattern, text_lower):
            return False, 'excluded'

    # Count buildout keyword matches
    buildout_score = 0
    for pattern in BUILDOUT_KEYWORDS:
        if re.search(pattern, text_lower):
            buildout_score += 1

    # No buildout signals at all
    if buildout_score == 0:
        return False, 'no_signal'

    # Confidence scoring
    has_company = company is not None
    has_mw = mw_capacity is not None

    # HIGH: Company + (Location or MW) + strong buildout signals
    if has_company and has_mw and buildout_score >= 3:
        return True, 'high'

    if has_company and buildout_score >= 4:
        return True, 'high'

    # MEDIUM: Company + some buildout signals
    if has_company and (has_mw or buildout_score >= 2):
        return True, 'medium'

    # LOW: Company only, or MW + strong signals without company
    if has_company and buildout_score >= 1:
        return True, 'low'

    if has_mw and buildout_score >= 3:
        return True, 'low'

    return False, 'weak_signal'


def get_text_snippet(text, max_chars=200):
    """Return a short text snippet for debugging/reference."""
    if not text:
        return ''
    return text[:max_chars].replace('\n', ' ').strip()


print("\u2705 All extraction functions defined")

## Step 3: Run Extraction Pipeline

Iterate over each candidate URL: fetch, extract, classify.

Includes rate limiting and error handling.

In [ ]:
# Output columns
OUTPUT_COLS = [
    'url', 'source_domain', 'date', 'company', 'location_city',
    'location_state', 'mw_capacity', 'target_completion_date',
    'is_buildout', 'confidence', 'extracted_text_snippet',
    'v2_organizations', 'v2_locations', 'v2_tone'
]

# Pipeline configuration
REQUEST_DELAY = 1.0  # seconds between requests (rate limiting)
MAX_URLS = None  # Set to int for testing; None = process all

results = []
errors = []
total = len(df)

if MAX_URLS:
    total = min(total, MAX_URLS)
    print(f"\U0001f504 Running limited pipeline on {total} URLs...")
else:
    print(f"\U0001f504 Running full pipeline on {total} URLs...")

for idx, row in df.head(total).iterrows():
    url = row[url_col]
    domain = row.get('SourceCommonName', row.get('source_domain', ''))
    gkg_date = str(row.get('DATE', row.get('date', '')))
    v2_orgs = str(row.get('V2Organizations', ''))
    v2_locs = str(row.get('V2Locations', ''))
    v2_tone = row.get('V2Tone', None)

    # Progress
    if (idx + 1) % 10 == 0 or idx == 0:
        print(f"\U0001f504 [{idx+1}/{total}] Fetching: {url[:80]}...")

    # Skip empty/malformed URLs
    if not url or not isinstance(url, str) or not url.startswith('http'):
        errors.append({'url': url, 'reason': 'invalid_url'})
        continue

    # Step 1: Fetch and extract article text
    text, method = extract_article_text(url)

    if text is None:
        errors.append({'url': url, 'reason': 'fetch_failed'})
        # Still try to classify from metadata if available
        text = ''

    # Step 2: Extract structured fields
    company = extract_company(v2_organizations=v2_orgs, text=text)
    mw_capacity = extract_mw(text)
    city, state = extract_location(text, v2_locations=v2_locs)
    target_date = extract_target_completion_date(text)

    # Step 3: Classify
    is_buildout, confidence = classify_buildout(text, company, mw_capacity)

    # Build result row
    result = {
        'url': url,
        'source_domain': domain,
        'date': gkg_date,
        'company': company,
        'location_city': city,
        'location_state': state,
        'mw_capacity': mw_capacity,
        'target_completion_date': target_date,
        'is_buildout': is_buildout,
        'confidence': confidence,
        'extracted_text_snippet': get_text_snippet(text),
        'v2_organizations': v2_orgs,
        'v2_locations': v2_locs,
        'v2_tone': v2_tone,
    }
    results.append(result)

    # Rate limiting
    time.sleep(REQUEST_DELAY)

# Build DataFrame from results
df_results = pd.DataFrame(results, columns=OUTPUT_COLS)
df_errors = pd.DataFrame(errors) if errors else pd.DataFrame()

print(f"\n\u2705 Pipeline complete!")
print(f"   Total processed: {len(df_results)}")
print(f"   Successful fetches: {df_results['extracted_text_snippet'].str.len().gt(0).sum()}")
print(f"   Fetch errors: {len(errors)}")
print(f"   Buildout classified: {df_results['is_buildout'].sum()}")

## Step 4: Summary Statistics

In [ ]:
# --- Overall counts ---
print("\n" + "="*60)
print("\U0001f4ca EXTRACTION SUMMARY")
print("="*60)

total_urls = len(df_results)
fetched = df_results['extracted_text_snippet'].str.len().gt(0).sum()
buildout_count = df_results['is_buildout'].sum()

print(f"\U0001f310 Total URLs processed: {total_urls}")
print(f"\u2705 Successfully fetched: {fetched} ({fetched/total_urls*100:.1f}%)")
print(f"\u274c Fetch failures: {total_urls - fetched} ({(total_urls-fetched)/total_urls*100:.1f}%)")
print(f"\U0001f3d7\ufe0f Classified as buildout: {buildout_count} ({buildout_count/total_urls*100:.1f}%)")

print(f"\n--- By Confidence Level ---")
if buildout_count > 0:
    buildout_df = df_results[df_results['is_buildout']]
    conf_counts = buildout_df['confidence'].value_counts()
    for conf in ['high', 'medium', 'low']:
        if conf in conf_counts.index:
            print(f"   {conf.title()}: {conf_counts[conf]}")

print(f"\n--- By Company ---")
if buildout_count > 0:
    company_counts = buildout_df['company'].fillna('unknown').value_counts()
    print(company_counts.to_string())

print(f"\n--- Has MW Capacity ---")
has_mw = df_results['mw_capacity'].notna().sum()
print(f"   Articles with MW mentioned: {has_mw}")
if has_mw > 0:
    print(f"   MW range: {df_results['mw_capacity'].min():.0f} - {df_results['mw_capacity'].max():.0f} MW")
    print(f"   Mean MW: {df_results['mw_capacity'].mean():.0f} MW")

print(f"\n--- Has Location ---")
has_loc = df_results['location_state'].notna().sum()
print(f"   Articles with US location: {has_loc}")

if has_loc > 0:
    print(f"\n--- Location Breakdown (Top States) ---")
    state_counts = df_results[df_results['location_state'].notna()]['location_state'].value_counts().head(10)
    print(state_counts.to_string())

## Step 5: Save Output

Save structured buildout events to CSV and track with DVC.

In [ ]:
# Ensure output directory exists
os.makedirs('data/raw', exist_ok=True)

# Save to CSV
output_path = 'data/raw/buildout_events_raw.csv'
df_results.to_csv(output_path, index=False)
print(f"\U0001f4be Saved {len(df_results)} rows to {output_path}")
print(f"   File size: {os.path.getsize(output_path):,} bytes")

# Save errors separately (if any)
if not df_errors.empty:
    errors_path = 'data/raw/buildout_fetch_errors.csv'
    df_errors.to_csv(errors_path, index=False)
    print(f"   Saved {len(df_errors)} fetch errors to {errors_path}")

# DVC tracking
print("\n\U0001f504 Running DVC add...")
try:
    result = subprocess.run(
        ['dvc', 'add', output_path],
        capture_output=True, text=True, check=True
    )
    print(result.stdout)

    # Push to remote
    result_push = subprocess.run(
        ['dvc', 'push', output_path + '.dvc'],
        capture_output=True, text=True
    )
    if result_push.returncode == 0:
        print("\u2705 DVC push successful")
    else:
        print(f"\u26a0\ufe0f DVC push issue (may need remote config): {result_push.stderr}")
except Exception as e:
    print(f"\u26a0\ufe0f DVC step skipped: {e}")
    print("   Run 'dvc add data/raw/buildout_events_raw.csv' manually.")

print(f"\n\u2705 Notebook complete! Output ready for gridstatus labeling (T6).")

## Summary

✅ **Article Extraction Complete**

- Loaded candidate URLs from `data/raw/buildout_candidates_gkg.csv`
- Fetched article text via trafilatura (primary) / newspaper3k (fallback)
- Extracted: company, location, MW capacity, target completion dates
- Classified each candidate as buildout (with confidence) or non-buildout
- Output: `data/raw/buildout_events_raw.csv`
- DVC tracked

**Next steps**:
- Run `notebooks/14-gridstatus-labeling.ipynb` to cross-reference with ISO interconnection queues
- Label buildout events as promise_kept / promise_broken / in_progress